<a href="https://colab.research.google.com/github/servantjoseph/Entropy_Balancing/blob/main/ACS_EBW_Boosted_code_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -----------------------------
# Main analysis (last on 9/3/2026)
# -----------------------------

ACS_STATE = {}

def ensure_acs_data():
    """Download the OpenIntro acs12 CSV if it is not already present."""
    if os.path.exists(DATA_PATH):
        return

    import requests
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = "https://www.openintro.org/data/csv/acs12.csv"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/csv,application/csv,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    with open(DATA_PATH, "wb") as f:
        f.write(response.content)

    print("Saved:", DATA_PATH)

# def ensure_acs_data():
#     """Download the OpenIntro acs12 CSV if it is not already present."""
#     if os.path.exists(DATA_PATH):
#         return
#     import urllib.request
#     url = "https://www.openintro.org/data/csv/acs12.csv"
#     urllib.request.urlretrieve(url, DATA_PATH)


def _prepare_acs_inputs():
    ensure_acs_data()
    df = pd.read_csv(DATA_PATH)
    df = df[(df["age"] >= 18) & df["income"].notna()].copy().reset_index(drop=True)
    df["log_income"] = np.log1p(df["income"].astype(float))
    N = len(df)

    X_main_df, X_compact_df = make_features(df)
    X_main_raw = X_main_df.to_numpy(dtype=float)
    X_comp_raw = X_compact_df.to_numpy(dtype=float)

    X_main_std, _, _, _ = standardize_train_target(X_main_raw, X_main_raw)
    X_main_std, keep_main = drop_zero_variance(X_main_std)
    main_names = [c for c, keep in zip(X_main_df.columns, keep_main) if keep]
    X_comp_std, _, _, _ = standardize_train_target(X_comp_raw, X_comp_raw)
    X_comp_std, keep_comp = drop_zero_variance(X_comp_std)
    comp_names = [c for c, keep in zip(X_compact_df.columns, keep_comp) if keep]

    mu_main = X_main_std.mean(axis=0)
    pair_prod = pairwise_products(X_comp_std[:, :8])
    pair_all = np.hstack([X_main_std, pair_prod])
    pair_all, keep_pair = drop_zero_variance(pair_all)
    mu_pair = pair_all.mean(axis=0)
    wt = np.ones(N) / N
    y = df["log_income"].to_numpy(dtype=float)
    y_income = df["income"].to_numpy(dtype=float)
    target_log = float(np.mean(y))
    target_income = float(np.mean(y_income))

    c = X_compact_df
    age_scaled = (c["age"].values - c["age"].mean()) / c["age"].std()
    hours_scaled = (c["hrs_work"].values - c["hrs_work"].mean()) / (c["hrs_work"].std() + 1e-8)
    commute_scaled = (c["commute"].values - c["commute"].mean()) / (c["commute"].std() + 1e-8)
    employed = c["employed"].values
    male = c["male"].values
    college = c["college"].values
    nonwhite = c["nonwhite"].values
    english = c["english"].values
    married = c["married"].values
    disabled = c["disabled"].values
    score = (
        0.12 * employed +
        0.10 * college +
        0.06 * male +
        0.06 * married -
        0.05 * disabled +
        0.04 * age_scaled +
        0.05 * hours_scaled -
        0.04 * commute_scaled +
        2.20 * (age_scaled > 0.70) * college +
        1.70 * (hours_scaled > 0.60) * employed +
        1.20 * (commute_scaled < -0.50) * male +
        1.00 * (age_scaled < -0.75) * (1 - english) -
        1.20 * nonwhite * (1 - english) +
        0.75 * employed * college * male
    )
    probs = stable_softmax(score)
    return {
        "N": N,
        "X_main_std": X_main_std,
        "X_comp_std": X_comp_std,
        "pair_all": pair_all,
        "mu_main": mu_main,
        "mu_pair": mu_pair,
        "wt": wt,
        "y": y,
        "y_income": y_income,
        "target_log": target_log,
        "target_income": target_income,
        "probs": probs,
        "main_names": main_names,
        "comp_names": comp_names,
    }


def _acs_worker_init(state):
    ACS_STATE.clear()
    ACS_STATE.update(state)


def _acs_one_rep(args):
    r, n_source = args
    s = ACS_STATE
    rng = np.random.default_rng(SEED + r)
    N = s["N"]
    idx = rng.choice(N, size=n_source, replace=False, p=s["probs"])
    Xs_main = s["X_main_std"][idx, :]
    Xs_comp = s["X_comp_std"][idx, :]
    ys = s["y"][idx]
    ys_income = s["y_income"][idx]
    mu_m = s["mu_main"]
    failures = {"main": 0, "pairwise": 0, "hybrid_main_init": 0, "hybrid_pairwise_init": 0}
    methods = []
    w_un = np.ones(n_source) / n_source
    q = w_un.copy()
    methods.append(("Biased source", w_un, None))
    try:
        w_main, lam_main = eb_fit(Xs_main, mu_m, q=q, max_iter=80, tol=1e-7, ridge=1e-7)
    except Exception:
        failures["main"] += 1
        w_main = w_un.copy()
    methods.append(("Main-effect EB", w_main, None))
    Xs_pair = s["pair_all"][idx, :]
    try:
        w_pair, lam_pair = eb_fit(Xs_pair, s["mu_pair"], q=q, max_iter=80, tol=1e-7, ridge=1e-7)
    except Exception:
        failures["pairwise"] += 1
        w_pair = w_main.copy()
    methods.append(("Fixed pairwise EB", w_pair, None))
    try:
        rs = int(rng.integers(0, 2**31 - 1))
        w_hyb, nt = hybrid(Xs_comp, s["X_comp_std"], Xs_main, mu_m, q0=w_main, B=100, nu=0.10, min_mass=0.001, interaction_depth=2, random_state=rs, min_ess_frac=0.10, score_tol=0.05)
    except Exception:
        failures["hybrid_main_init"] += 1
        w_hyb = w_main.copy()
        nt = 0
    methods.append(("EB-offset hybrid: main", w_hyb, nt))
    try:
        w_hyb_pair, ntp = hybrid(Xs_comp, s["X_comp_std"], Xs_pair, s["mu_pair"], q0=w_pair, B=100, nu=0.10, min_mass=0.001, interaction_depth=2, random_state=rs + 10000, min_ess_frac=0.10, score_tol=0.05)
    except Exception:
        failures["hybrid_pairwise_init"] += 1
        w_hyb_pair = w_pair.copy()
        ntp = 0
    methods.append(("EB-offset hybrid: pairwise", w_hyb_pair, ntp))
    rows = []
    for method, w, extra in methods:
        est = float(w @ ys)
        est_income = float(w @ ys_income)
        main_l2 = float(np.linalg.norm(Xs_main.T @ w - mu_m))
        validation_tv = validation_leaf_imbalance(Xs_comp, s["X_comp_std"], w, s["wt"])
        pair_l2 = float(np.linalg.norm(Xs_pair.T @ w - s["mu_pair"]))
        rows.append({
            "rep": r,
            "method": method,
            "estimate": est,
            "error": est - s["target_log"],
            "estimate_income": est_income,
            "error_income": est_income - s["target_income"],
            "main_l2": main_l2,
            "pair_l2": pair_l2,
            "validation_tv": validation_tv,
            "ess": effective_sample_size(w),
            "max_weight": float(np.max(w)),
            "num_trees": (extra if isinstance(extra, (int, np.integer)) else len(extra)) if extra is not None else np.nan,
        })
    return rows, failures


def run_analysis(R=200, n_source=450, n_jobs=None):
    import multiprocessing as mp
    state = _prepare_acs_inputs()
    if n_jobs is None:
        n_jobs = min(8, max(1, (os.cpu_count() or 2) - 1))
    args = [(r, n_source) for r in range(R)]
    raw_rows = []
    failures = {"main": 0, "pairwise": 0, "hybrid_main_init": 0, "hybrid_pairwise_init": 0}
    if n_jobs <= 1:
        _acs_worker_init(state)
        iterable = map(_acs_one_rep, args)
    else:
        pool = mp.Pool(processes=n_jobs, initializer=_acs_worker_init, initargs=(state,))
        iterable = pool.imap(_acs_one_rep, args, chunksize=5)
    try:
        for rows, fail in iterable:
            raw_rows.extend(rows)
            for k, v in fail.items():
                failures[k] += v
    finally:
        if n_jobs > 1:
            pool.close(); pool.join()
    raw = pd.DataFrame(raw_rows)
    summary = summarize_results(raw)
    income_rows = []
    for method, g in raw.groupby("method"):
        e = g["error_income"].values
        income_rows.append({
            "method": method,
            "bias_income": e.mean(),
            "rmse_income": math.sqrt(np.mean(e**2)),
            "mae_income": np.mean(np.abs(e)),
        })
    income_summary = pd.DataFrame(income_rows).sort_values("rmse_income")
    summary = summary.merge(income_summary, on="method", how="left")

    raw.to_csv(os.path.join(OUT_DIR, "acs_realdata_raw_rows.csv"), index=False)
    summary.to_csv(os.path.join(OUT_DIR, "acs_realdata_summary.csv"), index=False)
    meta = {
        "seed": SEED,
        "R": R,
        "n_source": n_source,
        "N_adult_pseudopopulation": state["N"],
        "target_mean_log_income": state["target_log"],
        "target_mean_income": state["target_income"],
        "main_feature_count": int(state["X_main_std"].shape[1]),
        "compact_feature_count": int(state["X_comp_std"].shape[1]),
        "failures": failures,
        "main_feature_names": state["main_names"],
        "compact_feature_names": state["comp_names"],
    }
    with open(os.path.join(OUT_DIR, "acs_realdata_metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    order = ["Biased source", "Main-effect EB", "Fixed pairwise EB", "EB-offset hybrid: main", "EB-offset hybrid: pairwise"]
    summary_ordered = summary.set_index("method").loc[order].reset_index()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["rmse"])
    plt.ylabel("RMSE for mean log income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_rmse_log_income.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["rmse_income"])
    plt.ylabel("RMSE for mean income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_rmse_income.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["validation_tv_mean"])
    plt.ylabel("Mean validation leaf total variation")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_validation_tv.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["ess_mean"])
    plt.ylabel("Mean effective sample size")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_ess.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    data = [raw.loc[raw["method"] == m, "error"].values for m in order]
    plt.boxplot(data, labels=order, showfliers=False)
    plt.axhline(0.0, linewidth=1)
    plt.ylabel("Error in mean log income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_error_boxplot.pdf"))
    plt.close()

    print("Summary")
    print(summary_ordered.to_string(index=False))
    print("Metadata")
    print(json.dumps(meta, indent=2)[:2000])
    return raw, summary, meta

if __name__ == "__main__":
    run_analysis(R=1000, n_source=450, n_jobs=8)


Summary
                    method  mean_estimate     bias  abs_bias     rmse      mae  main_l2_mean  pair_l2_mean  validation_tv_mean   ess_mean  max_weight_mean  bias_income  rmse_income   mae_income
             Biased source       7.081366 1.411836  1.411836 1.421231 1.411836  9.365242e-01      1.059973            0.112373 450.000000         0.002222 15045.258202 15127.285567 15045.258202
            Main-effect EB       5.711381 0.041851  0.041851 0.090902 0.073036  9.819798e-09      0.542584            0.014889 268.036932         0.013123 -1980.923767  2199.620966  2003.536960
         Fixed pairwise EB       5.744284 0.074754  0.074754 0.119881 0.100014  5.886811e-05      0.000074            0.007347 197.910268         0.022586  -128.362006  1092.153882   865.260854
    EB-offset hybrid: main       5.701080 0.031550  0.031550 0.090887 0.072863  2.680189e-09      0.485639            0.013357 261.149057         0.013661 -1716.877844  1985.520611  1762.760683
EB-offset hybrid: pair

/tmp/ipykernel_1439/812776193.py:283: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=order, showfliers=False)


# ACS EBW Boosted (Notebook)

This notebook is a direct conversion of `ACS_EBW_Boosted_code.py` into a runnable Jupyter notebook. It prepares the ACS data, constructs features, and computes biased sampling probabilities used for experiments with entropy balancing and hybrid tree-boosted entropy balancing.

In [ ]:
# ============================================================
# Load Entropy_Balancing repository
# ============================================================

import os
import sys

REPO_URL = "https://github.com/servantjoseph/Entropy_Balancing.git"
REPO_DIR = "/content/Entropy_Balancing"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Now import your module
#import entropy_common

Cloning into 'Entropy_Balancing'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 104 (delta 50), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 119.03 KiB | 975.00 KiB/s, done.
Resolving deltas: 100% (50/50), done.


In [ ]:
"""
Real-data-based ACS microdata analysis for hybrid tree-boosted entropy balancing.

Data: OpenIntro acs12 sample (2000 observations from the 2012 ACS). The script
uses adults age >= 18 as a finite pseudo-population, repeatedly draws biased
source samples using a nonlinear covariate-dependent selection rule, and compares
unweighted, main-effect EB, fixed pairwise EB, and EB-offset hybrid
tree-boosted EB.  The main-offset hybrid uses the fitted main-effect EB dual
parameters as the base offset; the pairwise-offset hybrid uses the fitted
pairwise EB dual parameters as the base offset and projects back to the same
main+pairwise hard constraints after each tree correction.

The EB fitting, pairwise-product construction, CART tree fitting, and EB-offset boosting routines are shared with the Kang--Schafer simulation implementation.
"""

import os
import math
import json
# from ACS_EBW_Boosted_code import BASE_DIR
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = os.getcwd()  # Use current working directory as base if __file__ is not defined
#BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' n globals() else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "acs12.csv")
OUT_DIR = BASE_DIR
SEED = 20260625


from entropy_common import (
    normalize_weights, effective_sample_size, sigmoid, eb_fit, eb_weights,
    pairwise_products, compact_leaf_ids_for_two, props, fit_balance_tree,
    hybrid, cell_props,validation_leaf_imbalance,summarize_results)

# -----------------------------
# Utility functions
# -----------------------------

def stable_softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    p = np.exp(z)
    return p / p.sum()


def weighted_mean(w, A):
    return np.asarray(w) @ np.asarray(A)


def normalize_weights(w):
    w = np.maximum(np.asarray(w, float), 1e-300)
    return w / w.sum()


def effective_sample_size(w):
    w = np.asarray(w, dtype=float)
    return 1.0 / np.sum(w * w)


def standardize_train_target(source_raw, target_raw):
    """Standardize source and target using target mean/sd."""
    mu = target_raw.mean(axis=0)
    sd = target_raw.std(axis=0)
    sd[sd < 1e-10] = 1.0
    return (source_raw - mu) / sd, (target_raw - mu) / sd, mu, sd


def drop_zero_variance(A, tol=1e-12):
    sd = A.std(axis=0)
    keep = sd > tol
    return A[:, keep], keep


def make_features(df):
    """Create raw feature matrices for main effects, compact pairwise basis, and tree search."""
    d = df.copy()
    # Numeric preprocessing for covariates only. Outcome is not used for weighting.
    d["hrs_work_imp"] = d["hrs_work"].fillna(0.0)
    d["hrs_work_missing"] = d["hrs_work"].isna().astype(float)
    d["time_to_work_imp"] = d["time_to_work"].fillna(0.0)
    d["time_to_work_missing"] = d["time_to_work"].isna().astype(float)
    d["lang_missing"] = d["lang"].isna().astype(float)
    d["edu_missing"] = d["edu"].isna().astype(float)
    d["lang"] = d["lang"].fillna("missing")
    d["edu"] = d["edu"].fillna("missing")

    # Include all covariate main effects. Drop first category for each factor.
    cat_cols = ["employment", "race", "gender", "citizen", "lang", "married", "edu", "disability", "birth_qrtr"]
    num_cols = ["age", "hrs_work_imp", "hrs_work_missing", "time_to_work_imp", "time_to_work_missing", "lang_missing", "edu_missing"]
    X_cat = pd.get_dummies(d[cat_cols], drop_first=True, dtype=float)
    X_num = d[num_cols].astype(float)
    X_main_df = pd.concat([X_num, X_cat], axis=1)

    # Compact features for pairwise products and tree search. Keep interpretable signals.
    compact = pd.DataFrame({
        "age": d["age"].astype(float),
        "hrs_work": d["hrs_work_imp"].astype(float),
        "commute": d["time_to_work_imp"].astype(float),
        "employed": (d["employment"] == "employed").astype(float),
        "male": (d["gender"] == "male").astype(float),
        "college": (d["edu"].isin(["college", "grad"])).astype(float),
        "grad": (d["edu"] == "grad").astype(float),
        "nonwhite": (d["race"] != "white").astype(float),
        "citizen": (d["citizen"] == "yes").astype(float),
        "english": (d["lang"] == "english").astype(float),
        "married": (d["married"] == "yes").astype(float),
        "disabled": (d["disability"] == "yes").astype(float),
    })
    return X_main_df, compact


In [ ]:
#update #interaction_depth=2;B=100;nu=0.100
#min_mass=0.025;tol=1e-7 tol=5e-7

# -----------------------------
# Main analysis
# -----------------------------

ACS_STATE = {}

def ensure_acs_data():
    """Download the OpenIntro acs12 CSV if it is not already present."""
    if os.path.exists(DATA_PATH):
        return

    import requests
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = "https://www.openintro.org/data/csv/acs12.csv"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/csv,application/csv,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()

    with open(DATA_PATH, "wb") as f:
        f.write(response.content)

    print("Saved:", DATA_PATH)

# def ensure_acs_data():
#     """Download the OpenIntro acs12 CSV if it is not already present."""
#     if os.path.exists(DATA_PATH):
#         return
#     import urllib.request
#     url = "https://www.openintro.org/data/csv/acs12.csv"
#     urllib.request.urlretrieve(url, DATA_PATH)


def _prepare_acs_inputs():
    ensure_acs_data()
    df = pd.read_csv(DATA_PATH)
    df = df[(df["age"] >= 18) & df["income"].notna()].copy().reset_index(drop=True)
    df["log_income"] = np.log1p(df["income"].astype(float))
    N = len(df)

    X_main_df, X_compact_df = make_features(df)
    X_main_raw = X_main_df.to_numpy(dtype=float)
    X_comp_raw = X_compact_df.to_numpy(dtype=float)

    X_main_std, _, _, _ = standardize_train_target(X_main_raw, X_main_raw)
    X_main_std, keep_main = drop_zero_variance(X_main_std)
    main_names = [c for c, keep in zip(X_main_df.columns, keep_main) if keep]
    X_comp_std, _, _, _ = standardize_train_target(X_comp_raw, X_comp_raw)
    X_comp_std, keep_comp = drop_zero_variance(X_comp_std)
    comp_names = [c for c, keep in zip(X_compact_df.columns, keep_comp) if keep]

    mu_main = X_main_std.mean(axis=0)
    pair_prod = pairwise_products(X_comp_std[:, :8])
    pair_all = np.hstack([X_main_std, pair_prod])
    pair_all, keep_pair = drop_zero_variance(pair_all)
    mu_pair = pair_all.mean(axis=0)
    wt = np.ones(N) / N
    y = df["log_income"].to_numpy(dtype=float)
    y_income = df["income"].to_numpy(dtype=float)
    target_log = float(np.mean(y))
    target_income = float(np.mean(y_income))

    c = X_compact_df
    age_scaled = (c["age"].values - c["age"].mean()) / c["age"].std()
    hours_scaled = (c["hrs_work"].values - c["hrs_work"].mean()) / (c["hrs_work"].std() + 1e-8)
    commute_scaled = (c["commute"].values - c["commute"].mean()) / (c["commute"].std() + 1e-8)
    employed = c["employed"].values
    male = c["male"].values
    college = c["college"].values
    nonwhite = c["nonwhite"].values
    english = c["english"].values
    married = c["married"].values
    disabled = c["disabled"].values
    score = (
        0.12 * employed +
        0.10 * college +
        0.06 * male +
        0.06 * married -
        0.05 * disabled +
        0.04 * age_scaled +
        0.05 * hours_scaled -
        0.04 * commute_scaled +
        2.20 * (age_scaled > 0.70) * college +
        1.70 * (hours_scaled > 0.60) * employed +
        1.20 * (commute_scaled < -0.50) * male +
        1.00 * (age_scaled < -0.75) * (1 - english) -
        1.20 * nonwhite * (1 - english) +
        0.75 * employed * college * male
    )
    probs = stable_softmax(score)
    return {
        "N": N,
        "X_main_std": X_main_std,
        "X_comp_std": X_comp_std,
        "pair_all": pair_all,
        "mu_main": mu_main,
        "mu_pair": mu_pair,
        "wt": wt,
        "y": y,
        "y_income": y_income,
        "target_log": target_log,
        "target_income": target_income,
        "probs": probs,
        "main_names": main_names,
        "comp_names": comp_names,
    }


def _acs_worker_init(state):
    ACS_STATE.clear()
    ACS_STATE.update(state)


def _acs_one_rep(args):
    r, n_source = args
    s = ACS_STATE
    rng = np.random.default_rng(SEED + r)
    N = s["N"]
    idx = rng.choice(N, size=n_source, replace=False, p=s["probs"])
    Xs_main = s["X_main_std"][idx, :]
    Xs_comp = s["X_comp_std"][idx, :]
    ys = s["y"][idx]
    ys_income = s["y_income"][idx]
    mu_m = s["mu_main"]
    failures = {"main": 0, "pairwise": 0, "hybrid_main_init": 0, "hybrid_pairwise_init": 0}
    methods = []
    w_un = np.ones(n_source) / n_source
    q = w_un.copy()
    methods.append(("Biased source", w_un, None))
    try:
        w_main, lam_main = eb_fit(Xs_main, mu_m, q=q, max_iter=80, tol=1e-7, ridge=1e-7)
    except Exception:
        failures["main"] += 1
        w_main = w_un.copy()
    methods.append(("Main-effect EB", w_main, None))
    Xs_pair = s["pair_all"][idx, :]
    try:
        w_pair, lam_pair = eb_fit(Xs_pair, s["mu_pair"], q=q, max_iter=80, tol=1e-7, ridge=1e-7)
    except Exception:
        failures["pairwise"] += 1
        w_pair = w_main.copy()
    methods.append(("Fixed pairwise EB", w_pair, None))
    try:
        rs = int(rng.integers(0, 2**31 - 1))
        w_hyb, nt = hybrid(Xs_comp, s["X_comp_std"], Xs_main, mu_m, q0=w_main, B=100, nu=0.10, min_mass=0.001, interaction_depth=3, random_state=rs, min_ess_frac=0.10, score_tol=0.05)
    except Exception:
        failures["hybrid_main_init"] += 1
        w_hyb = w_main.copy()
        nt = 0
    methods.append(("EB-offset hybrid: main", w_hyb, nt))
    try:
        w_hyb_pair, ntp = hybrid(Xs_comp, s["X_comp_std"], Xs_pair, s["mu_pair"], q0=w_pair, B=100, nu=0.10, min_mass=0.001, interaction_depth=3, random_state=rs + 10000, min_ess_frac=0.10, score_tol=0.05)
    except Exception:
        failures["hybrid_pairwise_init"] += 1
        w_hyb_pair = w_pair.copy()
        ntp = 0
    methods.append(("EB-offset hybrid: pairwise", w_hyb_pair, ntp))
    rows = []
    for method, w, extra in methods:
        est = float(w @ ys)
        est_income = float(w @ ys_income)
        main_l2 = float(np.linalg.norm(Xs_main.T @ w - mu_m))
        validation_tv = validation_leaf_imbalance(Xs_comp, s["X_comp_std"], w, s["wt"])
        pair_l2 = float(np.linalg.norm(Xs_pair.T @ w - s["mu_pair"]))
        rows.append({
            "rep": r,
            "method": method,
            "estimate": est,
            "error": est - s["target_log"],
            "estimate_income": est_income,
            "error_income": est_income - s["target_income"],
            "main_l2": main_l2,
            "pair_l2": pair_l2,
            "validation_tv": validation_tv,
            "ess": effective_sample_size(w),
            "max_weight": float(np.max(w)),
            "num_trees": (extra if isinstance(extra, (int, np.integer)) else len(extra)) if extra is not None else np.nan,
        })
    return rows, failures


def run_analysis(R=200, n_source=450, n_jobs=None):
    import multiprocessing as mp
    state = _prepare_acs_inputs()
    if n_jobs is None:
        n_jobs = min(8, max(1, (os.cpu_count() or 2) - 1))
    args = [(r, n_source) for r in range(R)]
    raw_rows = []
    failures = {"main": 0, "pairwise": 0, "hybrid_main_init": 0, "hybrid_pairwise_init": 0}
    if n_jobs <= 1:
        _acs_worker_init(state)
        iterable = map(_acs_one_rep, args)
    else:
        pool = mp.Pool(processes=n_jobs, initializer=_acs_worker_init, initargs=(state,))
        iterable = pool.imap(_acs_one_rep, args, chunksize=5)
    try:
        for rows, fail in iterable:
            raw_rows.extend(rows)
            for k, v in fail.items():
                failures[k] += v
    finally:
        if n_jobs > 1:
            pool.close(); pool.join()
    raw = pd.DataFrame(raw_rows)
    summary = summarize_results(raw)
    income_rows = []
    for method, g in raw.groupby("method"):
        e = g["error_income"].values
        income_rows.append({
            "method": method,
            "bias_income": e.mean(),
            "rmse_income": math.sqrt(np.mean(e**2)),
            "mae_income": np.mean(np.abs(e)),
        })
    income_summary = pd.DataFrame(income_rows).sort_values("rmse_income")
    summary = summary.merge(income_summary, on="method", how="left")

    raw.to_csv(os.path.join(OUT_DIR, "acs_realdata_raw_rows.csv"), index=False)
    summary.to_csv(os.path.join(OUT_DIR, "acs_realdata_summary.csv"), index=False)
    meta = {
        "seed": SEED,
        "R": R,
        "n_source": n_source,
        "N_adult_pseudopopulation": state["N"],
        "target_mean_log_income": state["target_log"],
        "target_mean_income": state["target_income"],
        "main_feature_count": int(state["X_main_std"].shape[1]),
        "compact_feature_count": int(state["X_comp_std"].shape[1]),
        "failures": failures,
        "main_feature_names": state["main_names"],
        "compact_feature_names": state["comp_names"],
    }
    with open(os.path.join(OUT_DIR, "acs_realdata_metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    order = ["Biased source", "Main-effect EB", "Fixed pairwise EB", "EB-offset hybrid: main", "EB-offset hybrid: pairwise"]
    summary_ordered = summary.set_index("method").loc[order].reset_index()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["rmse"])
    plt.ylabel("RMSE for mean log income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_rmse_log_income.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["rmse_income"])
    plt.ylabel("RMSE for mean income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_rmse_income.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["validation_tv_mean"])
    plt.ylabel("Mean validation leaf total variation")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_validation_tv.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    plt.bar(summary_ordered["method"], summary_ordered["ess_mean"])
    plt.ylabel("Mean effective sample size")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_ess.pdf"))
    plt.close()

    plt.figure(figsize=(7.4, 4.2))
    data = [raw.loc[raw["method"] == m, "error"].values for m in order]
    plt.boxplot(data, labels=order, showfliers=False)
    plt.axhline(0.0, linewidth=1)
    plt.ylabel("Error in mean log income")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acs_fig_error_boxplot.pdf"))
    plt.close()

    print("Summary")
    print(summary_ordered.to_string(index=False))
    print("Metadata")
    print(json.dumps(meta, indent=2)[:2000])
    return raw, summary, meta

if __name__ == "__main__":
    run_analysis(R=1000, n_source=450, n_jobs=8)


Summary
                    method  mean_estimate     bias  abs_bias     rmse      mae  main_l2_mean  pair_l2_mean  validation_tv_mean   ess_mean  max_weight_mean  bias_income  rmse_income   mae_income
             Biased source       7.081366 1.411836  1.411836 1.421231 1.411836  9.365242e-01      1.059973            0.112373 450.000000         0.002222 15045.258202 15127.285567 15045.258202
            Main-effect EB       5.711381 0.041851  0.041851 0.090902 0.073036  9.819798e-09      0.542584            0.014889 268.036932         0.013123 -1980.923767  2199.620966  2003.536960
         Fixed pairwise EB       5.744284 0.074754  0.074754 0.119881 0.100014  5.886811e-05      0.000074            0.007347 197.910268         0.022586  -128.362006  1092.153882   865.260854
    EB-offset hybrid: main       5.690872 0.021342  0.021342 0.095477 0.075978  2.025155e-09      0.375846            0.010470 243.804664         0.015143 -1181.799797  1592.130983  1349.358963
EB-offset hybrid: pair

/tmp/ipykernel_1439/2483837466.py:283: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=order, showfliers=False)


The probs variable is created in the _prepare_acs_inputs function. It's calculated by applying a stable_softmax function to a score that is derived from various demographic and employment features of the ACS data (e.g., age, hours worked, college education, gender, etc.). This score essentially quantifies the propensity or likelihood of an individual being sampled.

These probs are then used in the _acs_one_rep function within the line idx = rng.choice(N, size=n_source, replace=False, p=s["probs"]). Here, probs acts as the probability distribution for selecting n_source individuals from the N total individuals in the pseudo-population.

Even with very small probabilities, rng.choice(N, size=n_source, replace=False, p=s["probs"]) will still select n_source elements, as long as n_source is less than or equal to N and not all probabilities are zero.

When you provide the p argument to np.random.choice, it treats these values as relative probabilities. If they don't sum to 1, numpy internally normalizes them so they do sum to 1 before making the selection. So, p=[0.001,0.002,0.003,0.004,0.005] would be normalized to a distribution where the elements still have their relative likelihoods, but now sum to 1. Then, it proceeds to draw n_source distinct samples based on this normalized distribution.

In your simple example, rng.choice(5, size=3, replace=False, p=[0.001,0.002,0.003,0.004,0.005]) would indeed select 3 unique elements from the 5 available, with higher likelihood for those with larger (even if still small in absolute terms) relative probabilities.